# Sincronizzazione file

## Descrizione del progetto

### Azienda

**SyncEase** - Software per la gestione e la condivisione dei dati aziendali

SyncEase è una società tecnologica italiana specializzata nello sviluppo di soluzioni software per la gestione, la protezione e la condivisione dei dati aziendali. Nata come spin-off di un system integrator di Bologna, oggi conta un team di circa settanta persone tra sviluppo, infrastruttura e assistenza clienti.

Il catalogo comprende strumenti di replica e backup incrementale, agent per la sincronizzazione tra server on-premise e storage cloud e una console di monitoraggio degli archivi distribuiti. I clienti tipici sono studi professionali, aziende manifatturiere e strutture sanitarie che gestiscono grandi volumi di documenti e non possono tollerare perdite o versioni incoerenti dei file.

SyncEase lavora principalmente in Python e Go, con un modello di rilascio continuo e un forte orientamento all'affidabilità: ogni componente viene validato su scenari di carico reali prima di arrivare in produzione. L'azienda valorizza codice leggibile, documentato e testabile, considerandolo parte integrante del prodotto.

###Progetto

**Oggetto:** Prototipo motore di sincronizzazione tra due cartelle

Ciao,

ti scrivo perché abbiamo bisogno di un prototipo solido per una parte centrale del nostro agent di backup.

Oggi molti clienti mantengono allineate due cartelle (tipicamente un server locale e una destinazione di backup) con procedure manuali o script improvvisati. Il risultato lo vediamo ogni settimana nei ticket: file aggiornati che non arrivano a destinazione, duplicazioni, copie che durano ore quando il volume cresce.

Vorrei che sviluppassi in Python un sistema di sincronizzazione che, ricevuti due percorsi (source_folder e destination_folder), renda la destinazione una copia coerente della sorgente. In particolare deve:

- copiare i file nuovi, cioè assenti nella destinazione;
- aggiornare i file modificati, quando la versione in sorgente è più recente;
- rimuovere i file obsoleti, presenti solo nella destinazione.

Il punto che ci interessa davvero è la parte prestazionale: vogliamo capire quanto possiamo guadagnare sfruttando multithreading per le operazioni di lettura/scrittura e multiprocessing per lavorare in parallelo su sottogruppi di file. Motiva nei commenti perché hai scelto un meccanismo piuttosto che l'altro nei vari punti del flusso.

Prima di passarlo al team infrastruttura, mettilo alla prova su almeno tre scenari concreti: una cartella con pochi file, una con molti file e una con struttura annidata di sottocartelle.

Consegnami il link pubblico a un notebook Google Colab, con codice commentato e una lettura sintetica di quello che hai osservato nei test.

Grazie, Alessio Canduzzi - Engineering Manager, Data Protection, SyncEase

### Didattica

Il progetto richiede di costruire un motore di sincronizzazione unidirezionale tra due directory, con particolare attenzione alla parallelizzazione delle operazioni su file.

Ragiona prima sul confronto tra gli alberi di file (percorsi relativi, timestamp di modifica, eventualmente dimensione), poi sulla strategia di esecuzione parallela. Ricorda che le operazioni di I/O e quelle CPU-bound traggono vantaggio da strumenti differenti: la scelta va spiegata, non solo implementata.

Cura la robustezza del codice: gestione delle eccezioni sui singoli file, creazione delle directory mancanti, logging delle azioni eseguite. Il notebook deve essere leggibile da un collega che non conosce il codice.

### Consigli operativi

- **Separa il confronto dall'esecuzione.** Costruisci prima la lista delle azioni da compiere (copia, aggiornamento, rimozione) e solo dopo eseguila in parallelo: rende il codice testabile e il comportamento prevedibile.
- **Thread per I/O, processi per CPU.** Le copie di file sono dominate dall'I/O e beneficiano dei thread; la scansione o l'eventuale calcolo di hash su molti file può giustificare i processi. Motiva la scelta nei commenti.
- **Attenzione al GIL e all'overhead.** Avviare processi ha un costo: su pochi file il parallelismo può risultare più lento del codice sequenziale. Mostra questo effetto nei test invece di ignorarlo.
- **Genera dataset di prova via codice.** Crea le cartelle di test programmaticamente in Colab (file di dimensioni diverse, sottocartelle annidate) così i risultati sono riproducibili da chi legge.
- **Misura i tempi.** Confronta con time o timeit la versione sequenziale e quella parallela sui tre scenari richiesti: i numeri rendono credibili le tue conclusioni.
- **Gestisci gli errori sui singoli file.** Un file bloccato o senza permessi non deve interrompere l'intera sincronizzazione: raccogli gli errori e riportali a fine esecuzione.

## Obiettivi del progetto

1. Notebook Google Colab condiviso con link pubblico accessibile
2. Implementazione in Python di un sistema che sincronizza destination_folder rispetto a source_folder
3. Gestione di file nuovi (copia), file modificati (aggiornamento in base al timestamp) e file obsoleti (rimozione)
4. Uso di multithreading per le operazioni di lettura/scrittura dei file
5. Uso di multiprocessing per l'elaborazione parallela di sottogruppi di file
6. Codice commentato con spiegazione delle scelte implementative e del flusso del programma
7. Test su almeno 3 scenari: cartella con pochi file, cartella con molti file, cartella con sottocartelle
8. Breve commento finale sui risultati osservati nei test, con riferimento ai tempi di esecuzione

---

## Svolgimento del progetto

Il motore lavora in due fasi: prima confronta i due alberi e produce la lista delle azioni, poi la esegue. Per ogni fase usiamo lo strumento di parallelismo adatto al tipo di lavoro.

0. Inizializzazione ambiente
1. Scenari di prova generati via codice
2. Confronto tra gli alberi: la lista delle azioni
3. Esecuzione delle azioni
4. Il motore completo e la verifica del risultato
5. Errori sui singoli file e sulle cartelle
6. Misura dei tempi sui tre scenari
7. Conclusioni
8. Approfondimento oltre la consegna: una destinazione con latenza

## 0. Inizializzazione ambiente

Stampiamo la versione di Python e il numero di CPU disponibili: i tempi misurati più avanti vanno letti in rapporto a questi due dati. Stampiamo anche se il GIL è attivo, perché le letture della sezione 6 lo danno per scontato.

In [1]:
# importiamo le librerie e fissiamo le costanti usate in tutto il notebook

import errno
import hashlib
import logging
import multiprocessing
import os
import platform
import random
import shutil
import stat
import sys
import tempfile
import threading
import time
from collections import Counter
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from functools import partial, wraps
from pathlib import Path

import pandas as pd

RANDOM_STATE = 42

# One process per available CPU.
CPU_COUNT = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
PROCESS_WORKERS = CPU_COUNT
# More threads than CPUs: while some threads wait for a write, the others keep working.
THREAD_WORKERS = 8
# From 3.13 CPython also comes in a free-threaded build, without the GIL. Every reading of the
# timings assumes the standard build, so the runtime in use is printed below.
GIL_ENABLED = sys._is_gil_enabled() if hasattr(sys, "_is_gil_enabled") else True

# With fork the worker processes can use the functions defined in this notebook.
FORK_CONTEXT = multiprocessing.get_context("fork")

# Test folders go in the system temporary folder and are recreated at every run.
WORKSPACE = Path(tempfile.gettempdir()) / "sync_lab"
shutil.rmtree(WORKSPACE, ignore_errors=True)
WORKSPACE.mkdir(parents=True)

# Every executed action goes to a log file, written only by the main process: the workers
# return what they did and the main process logs it, so lines from different processes
# never get mixed up.
LOG_PATH = WORKSPACE / "sync.log"
logger = logging.getLogger("sync")
logger.setLevel(logging.INFO)
logger.propagate = False
for old_handler in list(logger.handlers):
    logger.removeHandler(old_handler)
    old_handler.close()
log_handler = logging.FileHandler(LOG_PATH, mode="w")
log_handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
logger.addHandler(log_handler)

print(f"Python {platform.python_version()} - GIL enabled: {GIL_ENABLED} - usable CPUs: {CPU_COUNT} - "
      f"process workers: {PROCESS_WORKERS} - thread workers: {THREAD_WORKERS}")

Python 3.12.15 - GIL enabled: True - usable CPUs: 2 - process workers: 2 - thread workers: 8


## 1. Scenari di prova generati via codice

**Obiettivo.** Costruire in modo riproducibile le tre cartelle richieste (pochi file, molti file, sottocartelle annidate), con una destinazione che contenga tutti e tre i casi da sincronizzare.

Ogni scenario ha una cartella `source` e una cartella `backup`, cioè la destinazione com'era all'ultima sincronizzazione. Il backup lo costruiamo partendo dalla sorgente:

- un file su dieci manca, quindi va copiato;
- un altro file su dieci c'è, ma in una versione più vecchia: contenuto diverso, stessa dimensione, timestamp indietro di un'ora. Va aggiornato;
- in più ci sono file obsoleti, uno ogni venti file della sorgente, in cartelle che nella sorgente non esistono. Vanno rimossi, e con loro le cartelle che restano vuote;
- gli altri file sono identici, timestamp compreso, e non si toccano.

Il file è nominato in base al suo ruolo (`new_`, `modified_`, `same_`, `old_`), così la lista delle azioni e il log si controllano più facilmente. Il motore non legge mai il nome, guarda solo se il file c'è e che timestamp ha.

Prima di ogni prova ricreiamo la destinazione di lavoro copiando `backup`, così ogni misura parte dallo stesso stato.

In [2]:
# definiamo gli scenari e le funzioni che li generano in modo riproducibile

SCENARIOS = {
    "few_files": {"files": 10, "depth": 0, "size_kb": (256, 4096)},
    "many_files": {"files": 5000, "depth": 0, "size_kb": (1, 64)},
    "nested_folders": {"files": 2000, "depth": 6, "size_kb": (1, 256)},
}

# Out of every CHANGE_PERIOD (10) source files:
#   - the first is left out of the backup     -> the sync has to copy it (new file)
#   - the second is saved with older content  -> the sync has to update it (modified file)
#   - the others are saved unchanged          -> the sync has to leave them alone
# In addition, for every OBSOLETE_PERIOD (20) source files the backup gets one file that the
# source does not have -> the sync has to delete it (obsolete file).
# A fixed pattern instead of random choices makes sure all three actions show up in every
# scenario, even the one with only ten files.
CHANGE_PERIOD = 10
OBSOLETE_PERIOD = 20

# Timestamps are set explicitly, in nanoseconds: files written one after the other would get
# timestamps that depend on the speed of the machine.
BASE_MTIME_NS = 1_700_000_000 * 10**9
HOUR_NS = 3600 * 10**9


def write_file(path, data, mtime_ns):
    """Write data to path, creating the missing folders, and set its modification time."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(data)
    os.utime(path, ns=(mtime_ns, mtime_ns))


def random_relative_path(rng, depth, name):
    """Return a relative path for name, placed between 0 and depth folders below the root.

    Each level picks one of three folders, so a depth of 6 yields a tree of several hundred
    folders with files at every level.
    """
    folders = [f"dir_{rng.randrange(3)}" for _ in range(rng.randint(0, depth))]
    return Path(*folders, name)


def build_scenario(name, files, depth, size_kb):
    """Create WORKSPACE/name/source and WORKSPACE/name/backup and return their summary."""
    rng = random.Random(RANDOM_STATE)
    root = WORKSPACE / name
    source, backup = root / "source", root / "backup"
    for index in range(files):
        role = index % CHANGE_PERIOD
        # The name is there for the reader: the sync never looks at it.
        prefix = "new" if role == 0 else "modified" if role == 1 else "same"
        relative = random_relative_path(rng, depth, f"{prefix}_{index:05d}.bin")
        data = rng.randbytes(rng.randint(*size_kb) * 1024)
        mtime_ns = BASE_MTIME_NS + index * 10**9
        write_file(source / relative, data, mtime_ns)
        if role == 0:
            continue
        if role == 1:
            # Same size, different content, older timestamp: only the timestamp tells them apart.
            write_file(backup / relative, rng.randbytes(len(data)), mtime_ns - HOUR_NS)
        else:
            write_file(backup / relative, data, mtime_ns)
    for index in range(max(1, files // OBSOLETE_PERIOD)):
        relative = Path(f"retired_{index % 2}") / random_relative_path(rng, depth, f"old_{index:05d}.bin")
        write_file(backup / relative, rng.randbytes(1024), BASE_MTIME_NS)
    return {"scenario": name, **folder_summary(source, "source"), **folder_summary(backup, "backup")}


def folder_summary(root, label):
    """Count files, folders and megabytes below root, with keys prefixed by label."""
    files = folders = size = 0
    for current, subfolders, names in os.walk(root):
        folders += len(subfolders)
        files += len(names)
        size += sum(os.path.getsize(os.path.join(current, entry)) for entry in names)
    return {f"{label}_files": files, f"{label}_folders": folders, f"{label}_mb": round(size / 2**20, 1)}


def prepare_destination(name):
    """Recreate WORKSPACE/name/destination as a fresh copy of the backup and return the paths."""
    root = WORKSPACE / name
    destination = root / "destination"
    shutil.rmtree(destination, ignore_errors=True)
    # copy2 keeps the timestamps, so the destination starts exactly as the backup was.
    shutil.copytree(root / "backup", destination, copy_function=shutil.copy2)
    return root / "source", destination


summary = pd.DataFrame([build_scenario(name, **spec) for name, spec in SCENARIOS.items()])
summary.set_index("scenario")

,source_files,source_folders,source_mb,backup_files,backup_folders,backup_mb
scenario,,,,,,
few_files,10,0,18.0,10,1,17.7
many_files,5000,0,155.2,4750,2,139.8
nested_folders,2000,572,247.4,1900,681,222.5


I tre scenari coprono i casi chiesti dalla consegna: 10 file da qualche megabyte l'uno, 5000 file piccoli in un'unica cartella e 2000 file sparsi in 572 sottocartelle, fino a sei livelli di profondità. Nello scenario con molti file il backup ha 4750 file, perché mancano i 500 nuovi e ci sono in più i 250 obsoleti.

## 2. Confronto tra gli alberi: la lista delle azioni

**Obiettivo.** Confrontare sorgente e destinazione e produrre la lista delle azioni `copy`, `update` e `delete`, senza toccare nessun file.

Le regole sono quelle della consegna, applicate al percorso relativo di ogni file:

| caso | azione |
| --- | --- |
| il file è in sorgente ma non in destinazione | `copy` |
| il file è in entrambe e in sorgente ha un timestamp di modifica più recente | `update` |
| il file è solo in destinazione | `delete` |

**Per decidere guardiamo solo il timestamp, la dimensione no.** Nella sezione 1 i file modificati hanno apposta la stessa dimensione dell'originale: un controllo sulla dimensione non li vedrebbe, il timestamp sì. Per ogni file il confronto fa quindi due sole chiamate a `os.stat`, una in sorgente e una in destinazione, senza leggerne il contenuto. L'MD5 che serve a dimostrare che la destinazione è identica byte per byte lo calcoliamo a parte, nel codice di prova della sezione 4: dentro il confronto obbligherebbe a leggere per intero ogni file sorgente, e i tempi della sezione 6 misurerebbero la prova insieme al motore.

Il tipo si controlla in modo esplicito. Se in destinazione, al posto di un file della sorgente, c'è una cartella con lo stesso nome, o un file al posto di una cartella del suo percorso, il file va copiato; la sezione 3 spiega in che ordine si risolve il conflitto.

**Il confronto lo facciamo fare ai processi.** Per ogni file bisogna leggere i metadati e confrontarli, e quasi tutto questo lavoro lo fa l'interprete Python. Con i thread il GIL ne lascerebbe girare uno alla volta; con più processi, ognuno col suo interprete e il suo gruppo di file.

In [3]:
# definiamo il confronto, scritto per lavorare su un sottogruppo di file alla volta

def error_message(error):
    """Describe an OSError in one line, the way the report and the log show it."""
    return f"{type(error).__name__}: {error.strerror}"


def check_folders(source, destination):
    """Raise ValueError unless source is an existing folder that can be synced into destination.

    A missing source would list no files, so every file of the destination would look
    obsolete and be planned for deletion: one mistyped path would empty the backup. The
    two folders must also not overlap, or the sync would write into, or delete from, the
    tree it is reading.
    """
    source, destination = Path(source).resolve(), Path(destination).resolve()
    if not source.exists():
        raise ValueError(f"source folder does not exist: {source}")
    if not source.is_dir():
        raise ValueError(f"source is not a folder: {source}")
    if destination.exists() and not destination.is_dir():
        raise ValueError(f"destination is not a folder: {destination}")
    # is_relative_to is also true for two equal paths, so this covers the same folder too.
    if source.is_relative_to(destination) or destination.is_relative_to(source):
        raise ValueError(f"source and destination overlap: {source}, {destination}")


def list_files(root):
    """Return the set of file paths below root, relative to it and with forward slashes."""
    return {Path(current, entry).relative_to(root).as_posix()
            for current, _, names in os.walk(root) for entry in names}


def analyze_group(source, destination, relative_paths):
    """Compare a subgroup of source files with the destination, reading only their metadata.

    This is the worker handed to the pool, so it only receives and returns plain data and
    does not use shared state. It returns the actions for the subgroup and the errors met,
    one per file: a file that cannot be read is reported and skipped, the rest of the
    subgroup goes on.
    """
    actions, errors = [], []
    for relative in relative_paths:
        try:
            source_mtime = os.stat(source / relative).st_mtime_ns
            try:
                target = os.stat(destination / relative)
            except (FileNotFoundError, NotADirectoryError):
                # NotADirectoryError: on the way to this file the destination has a file where
                # the source has a folder. That file is obsolete and is deleted before the copy.
                actions.append(("copy", relative))
                continue
            if stat.S_ISDIR(target.st_mode):
                # The destination has a folder with this file's name: its timestamp says nothing
                # about the file. The folder content is obsolete and is deleted before the copy.
                actions.append(("copy", relative))
            # Integer nanoseconds, the precision copy2 writes back: float seconds would round
            # the last digits and could make two different timestamps look equal.
            elif source_mtime > target.st_mtime_ns:
                actions.append(("update", relative))
        except OSError as error:
            errors.append(("analyze", relative, error_message(error)))
    return actions, errors


def run_in_groups(worker, items, mode):
    """Split items into subgroups and run worker on each of them, in the requested mode.

    mode is "sequential" (one group, current process), "threads" or "processes". The
    subgroups are dealt round-robin, so that large and small files spread evenly across
    the workers instead of clustering in alphabetical runs.
    """
    if mode == "sequential":
        return [worker(items)]
    if mode == "threads":
        executor = ThreadPoolExecutor(max_workers=THREAD_WORKERS, thread_name_prefix="thread")
        workers = THREAD_WORKERS
    elif mode == "processes":
        executor = ProcessPoolExecutor(max_workers=PROCESS_WORKERS, mp_context=FORK_CONTEXT)
        workers = PROCESS_WORKERS
    else:
        raise ValueError(f"unknown mode: {mode}")
    groups = [items[start::workers] for start in range(workers)]
    with executor:
        return list(executor.map(worker, groups))


def plan_sync(source, destination, mode="processes"):
    """Build the list of actions that turns destination into a copy of source.

    The two folders are checked before anything is listed, so an invalid pair raises
    before any deletion is planned. Listing the two trees is quick and stays in the main
    process; the per-file comparison is split into subgroups. Obsolete files come from
    the difference of the two listings.
    """
    check_folders(source, destination)
    source_files = list_files(source)
    destination_files = list_files(destination)
    # Each comparison is two stat calls wrapped in interpreter work. With processes the
    # subgroups really run at the same time; with threads the GIL lets only one of them run
    # at a time, apart from the moments spent inside stat.
    results = run_in_groups(partial(analyze_group, source, destination), sorted(source_files), mode)
    actions = [action for group_actions, _ in results for action in group_actions]
    actions += [("delete", relative) for relative in sorted(destination_files - source_files)]
    errors = [error for _, group_errors in results for error in group_errors]
    for _, relative, message in errors:
        logger.error("[analysis] %s failed: %s", relative, message)
    return {"actions": actions, "errors": errors}


source, destination = prepare_destination("few_files")
plan = plan_sync(source, destination)
print(f"Actions planned: {dict(Counter(kind for kind, _ in plan['actions']))}")
for kind, relative in plan["actions"]:
    print(f"  {kind:<7} {relative}")

Actions planned: {'update': 1, 'copy': 1, 'delete': 1}
  update  modified_00001.bin
  copy    new_00000.bin
  delete  retired_0/old_00000.bin


Sullo scenario con pochi file la lista contiene proprio i tre casi costruiti nella sezione 1: un file da copiare, uno da aggiornare e uno obsoleto da rimuovere. Gli altri sette file sono identici e non compaiono, quindi **il confronto non produce azioni inutili**.

Fin qui nessun file è stato toccato: la lista si può stampare e controllare prima di eseguirla. La esegue la sezione 3. Sulla stessa lista si regge anche la prova a vuoto della sezione 5.

## 3. Esecuzione delle azioni

**Obiettivo.** Eseguire in parallelo la lista delle azioni, creando le cartelle mancanti, scrivendo ogni azione nel log e raccogliendo gli errori senza fermarsi.

**L'esecuzione la affidiamo ai thread.** Una copia legge e scrive. Se la scrittura deve aspettare qualcosa che non è la CPU (un disco lento, una cartella di rete, uno storage cloud), durante l'attesa il thread rilascia il GIL e un altro thread può partire con la copia successiva. In più i thread costano poco da avviare e condividono la memoria del processo principale.

Le azioni girano in due turni: prima le rimozioni, poi copie e aggiornamenti, come fa rsync con l'opzione `--delete-before`. Con quest'ordine un file e una cartella con lo stesso nome non si incontrano mai, perché quello obsoleto sparisce prima che arrivi quello nuovo; il prezzo è che i worker partono due volte, e con i processi anche il loro avvio si paga due volte.

Prima di ogni copia creiamo le cartelle che mancano. Fra i due turni eliminiamo quelle rimaste vuote dopo le rimozioni, partendo dalla più profonda. Una cartella che non si riesce a rimuovere diventa un errore da riportare, come quello di un file.

In [4]:
# definiamo l'esecuzione delle azioni, anch'essa per sottogruppi

def worker_name():
    """Return the name of the thread or process running the caller, as written in the log."""
    process = multiprocessing.current_process().name
    return threading.current_thread().name if process == "MainProcess" else process


def execute_group(source, destination, actions, copy_function=shutil.copy2):
    """Carry out a subgroup of actions and return the ones done, the errors met and the worker.

    Each file is handled inside its own try block, so a failure is recorded and the
    loop moves on to the next file. copy_function is replaced only in section 8, to
    model a slower destination.
    """
    done, errors = [], []
    for kind, relative in actions:
        target = destination / relative
        try:
            if kind == "delete":
                target.unlink()
            elif target.is_dir():
                # Only left when the deletions could not empty a folder with this file's name:
                # copy2 would write the file inside the folder instead of in its place.
                raise IsADirectoryError(errno.EISDIR, os.strerror(errno.EISDIR), str(target))
            else:
                # exist_ok makes the creation safe when two threads need the same folder.
                target.parent.mkdir(parents=True, exist_ok=True)
                # copy2 also copies the timestamp, so after the copy source and destination
                # carry the same one: "equal" then means "same version", and the rule
                # "newer in source" compares two versions instead of a version and a copy date.
                copy_function(source / relative, target)
            done.append((kind, relative))
        except OSError as error:
            errors.append((kind, relative, error_message(error)))
    return done, errors, worker_name()


def remove_empty_folders(destination, source):
    """Remove the destination folders that are empty and do not exist in the source.

    The walk goes bottom-up, so a folder emptied by the removal of its subfolders is
    seen as empty in the same pass. A folder that cannot be removed is reported and
    skipped, like a file. Return the number of folders removed and the errors met.
    """
    removed, errors = 0, []
    for current, _, _ in os.walk(destination, topdown=False):
        folder = Path(current)
        if folder == destination:
            continue
        relative = folder.relative_to(destination)
        try:
            if not any(folder.iterdir()) and not (source / relative).is_dir():
                folder.rmdir()
                removed += 1
        except OSError as error:
            errors.append(("rmdir", relative.as_posix(), error_message(error)))
    return removed, errors


def execute_plan(source, destination, actions, mode="threads", copy_function=shutil.copy2):
    """Run the actions in the requested mode, log each outcome and return the totals.

    The deletions run first, then the folders they leave empty are removed, and only then
    the copies and updates: a file and a folder with the same name never meet, because the
    obsolete one is gone before the new one is written.
    """
    run_group = partial(execute_group, source, destination, copy_function=copy_function)
    deletions = [action for action in actions if action[0] == "delete"]
    writes = [action for action in actions if action[0] != "delete"]
    # Copying a file means waiting for the destination. While one thread waits, another one
    # can start the next copy, and threads cost much less than processes to start.
    results = run_in_groups(run_group, deletions, mode)
    removed_folders, folder_errors = remove_empty_folders(destination, source)
    # The folder errors come from the main process and go through the same log as the others.
    results.append(([], folder_errors, worker_name()))
    results += run_in_groups(run_group, writes, mode)
    done, errors, workers = [], [], Counter()
    for group_done, group_errors, worker in results:
        if group_done:
            workers[worker] += len(group_done)
        for kind, relative in group_done:
            logger.info("[%s] %s %s", worker, kind, relative)
        for kind, relative, message in group_errors:
            logger.error("[%s] %s %s failed: %s", worker, kind, relative, message)
        done += group_done
        errors += group_errors
    return {"done": done, "errors": errors, "workers": workers, "removed_folders": removed_folders}


outcome = execute_plan(source, destination, plan["actions"])
print(f"Actions done: {dict(Counter(kind for kind, _ in outcome['done']))} - "
      f"errors: {len(outcome['errors'])} - empty folders removed: {outcome['removed_folders']}")
print("Last lines of the log:")
print("".join(LOG_PATH.read_text().splitlines(keepends=True)[-3:]), end="")

Actions done: {'delete': 1, 'update': 1, 'copy': 1} - errors: 0 - empty folders removed: 1
Last lines of the log:
2026-10-14 20:57:55,790 INFO [thread_0] delete retired_0/old_00000.bin
2026-10-14 20:57:55,790 INFO [thread_0] update modified_00001.bin
2026-10-14 20:57:55,790 INFO [thread_1] copy new_00000.bin


Le tre azioni pianificate vanno a buon fine senza errori, e la cartella `retired_0`, rimasta vuota dopo la rimozione del file obsoleto, viene eliminata.

Nel log c'è un'azione per riga, con data, ora e il worker che l'ha eseguita, così dopo una sincronizzazione si può ricostruire cosa è successo. La rimozione compare prima di copia e aggiornamento, nell'ordine dei due turni. **Il log lo scrive solo il processo principale**, a partire dai risultati che gli restituiscono i worker: per questo le righe non si mescolano neanche quando l'esecuzione è parallela.

## 4. Il motore completo e la verifica del risultato

**Obiettivo.** Mettere insieme confronto ed esecuzione in un'unica funzione `sync_folders(source_folder, destination_folder)` e verificare che la destinazione diventi davvero una copia coerente della sorgente.

Facciamo due prove indipendenti:

- check del contenuto: l'MD5 di ogni file in destinazione deve coincidere con quello calcolato sulla sorgente dal codice di prova;
- check dei residui: se la sincronizzazione avesse lasciato anche una sola differenza, un secondo confronto troverebbe ancora qualcosa da fare, e invece non deve trovare nessuna azione.

Le due prove sono degli `assert`, quindi se una non passa il notebook si ferma.

In [5]:
# riuniamo confronto ed esecuzione e verifichiamo il risultato su ogni scenario

ACTION_KINDS = ["copy", "update", "delete"]
DIGEST_CHUNK = 1024 * 1024


def sync_folders(source_folder, destination_folder, analysis_mode="processes", execution_mode="threads",
                 dry_run=False):
    """Make destination_folder a coherent copy of source_folder and return a report.

    The comparison runs first and touches nothing; only its list of actions is then
    executed, and with dry_run it is returned without being executed. An invalid pair of
    folders raises ValueError before anything is planned; the errors on single files of
    both phases are collected and returned together at the end.
    """
    source_folder, destination_folder = Path(source_folder), Path(destination_folder)
    plan = plan_sync(source_folder, destination_folder, analysis_mode)
    report = {"planned": Counter(kind for kind, _ in plan["actions"]), "errors": plan["errors"]}
    if dry_run:
        return {**report, "actions": plan["actions"]}
    destination_folder.mkdir(parents=True, exist_ok=True)
    outcome = execute_plan(source_folder, destination_folder, plan["actions"], execution_mode)
    return {**report,
            "done": Counter(kind for kind, _ in outcome["done"]),
            "removed_folders": outcome["removed_folders"],
            "workers": outcome["workers"],
            "errors": plan["errors"] + outcome["errors"]}


def file_digest(path):
    """Return the MD5 digest of a file, read in chunks so that large files fit in memory."""
    digest = hashlib.md5()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(DIGEST_CHUNK), b""):
            digest.update(chunk)
    return digest.hexdigest()


def tree_digests(root):
    """Return the MD5 digest of every file below root, keyed by its relative path."""
    return {relative: file_digest(root / relative) for relative in list_files(root)}


verification_rows, actions_per_worker = [], {}
for name in SCENARIOS:
    source, destination = prepare_destination(name)
    report = sync_folders(source, destination)
    # The digests belong to the test, not to the engine: computed here, they check its
    # result without weighing on the comparison timed in section 6.
    source_digests, copied_digests = tree_digests(source), tree_digests(destination)
    second_pass = plan_sync(source, destination)["actions"]
    assert copied_digests == source_digests, f"{name}: destination content differs from source"
    assert not second_pass, f"{name}: a second pass still finds {len(second_pass)} actions"
    verification_rows.append({"scenario": name, **{kind: report["done"][kind] for kind in ACTION_KINDS},
                              "removed_folders": report["removed_folders"],
                              "errors": len(report["errors"]),
                              "identical_files": len(copied_digests),
                              "second_pass_actions": len(second_pass)})
    actions_per_worker[name] = report["workers"]

for name, workers in actions_per_worker.items():
    print(f"{name} - actions per worker: {dict(sorted(workers.items()))}")
pd.DataFrame(verification_rows).set_index("scenario")

few_files - actions per worker: {'thread_0': 2, 'thread_1': 1}
many_files - actions per worker: {'thread_0': 157, 'thread_1': 157, 'thread_2': 156, 'thread_3': 187, 'thread_4': 156, 'thread_5': 156, 'thread_6': 156, 'thread_7': 125}
nested_folders - actions per worker: {'thread_0': 88, 'thread_1': 63, 'thread_2': 63, 'thread_3': 62, 'thread_4': 62, 'thread_5': 62, 'thread_6': 50, 'thread_7': 50}


,copy,update,delete,removed_folders,errors,identical_files,second_pass_actions
scenario,,,,,,,
few_files,1,1,1,1,0,10,0
many_files,500,500,250,2,0,5000,0
nested_folders,200,200,100,142,0,2000,0


Su tutti e tre gli scenari il motore esegue le azioni attese: 500 copie, 500 aggiornamenti e 250 rimozioni con molti file, 200, 200 e 100 con le sottocartelle. **Le due prove passano**: le impronte dei 5000 e dei 2000 file in destinazione coincidono con quelle della sorgente, e il secondo passaggio non trova azioni. Se il motore avesse saltato un file nuovo, una versione vecchia o un file obsoleto, il secondo confronto lo avrebbe trovato.

Le righe sopra la tabella mostrano come si divide il lavoro fra i thread: ogni turno distribuisce le sue azioni a rotazione in otto sottogruppi, che differiscono al massimo di un'azione, ma i totali per thread non sono uguali, perché le rimozioni sono rapide e un thread che finisce presto il suo sottogruppo prende anche il successivo.

Nello scenario annidato vengono rimosse anche 142 cartelle rimaste vuote. Senza questo passaggio la destinazione si terrebbe una struttura di cartelle che nella sorgente non c'è più.

In [6]:
# costruiamo il caso che i tre scenari non contengono: un file e una cartella con lo stesso nome

conflict_root = WORKSPACE / "name_conflict"
source, destination = conflict_root / "source", conflict_root / "destination"
# In the source "reports" is a folder and "notes" a file; in the destination it is the
# other way round, so both names have to change type.
write_file(source / "reports" / "q1.bin", b"quarterly report", BASE_MTIME_NS)
write_file(source / "notes", b"notes", BASE_MTIME_NS)
write_file(destination / "reports", b"old report", BASE_MTIME_NS)
write_file(destination / "notes" / "draft.bin", b"draft", BASE_MTIME_NS)

report = sync_folders(source, destination)
second_pass = plan_sync(source, destination)["actions"]
assert tree_digests(destination) == tree_digests(source), "name conflict: destination differs from source"
assert not second_pass, f"name conflict: a second pass still finds {len(second_pass)} actions"
print(f"Planned: {dict(report['planned'])} - errors: {len(report['errors'])} - "
      f"second pass actions: {len(second_pass)}")
print(f"Destination files: {sorted(list_files(destination))}")

Planned: {'copy': 2, 'delete': 2} - errors: 0 - second pass actions: 0
Destination files: ['notes', 'reports/q1.bin']


Il motore risolve anche un caso che i tre scenari non contengono, un file e una cartella con lo stesso nome: `reports` passa da file a cartella e `notes` da cartella a file, senza errori, e il secondo passaggio non trova azioni. **Il conflitto si risolve perché le rimozioni vengono prima delle copie** (sezione 3). Se girassero insieme, la copia di `reports/q1.bin` potrebbe trovare ancora al suo posto il file `reports` e fallire, e la copia di `notes` potrebbe trovare ancora la cartella.

## 5. Errori sui singoli file e sulle cartelle

**Obiettivo.** Mostrare che un file non disponibile non blocca la sincronizzazione: l'errore viene raccolto, gli altri file vengono sincronizzati e il riepilogo finale lo riporta. Mostrare anche che una coppia di cartelle sbagliata, invece, la ferma prima che tocchi la destinazione.

Simuliamo il caso più comune su un sistema in uso: un file che c'era al momento del confronto sparisce prima della copia, perché qualcuno l'ha spostato o cancellato.

In [7]:
# facciamo sparire un file fra il confronto e l'esecuzione, come farebbe un utente

source, destination = prepare_destination("many_files")
plan = plan_sync(source, destination)
vanished = next(relative for kind, relative in plan["actions"] if kind == "copy")
removed_copy = WORKSPACE / "vanished.bin"
shutil.move(source / vanished, removed_copy)

outcome = execute_plan(source, destination, plan["actions"])
print(f"Actions planned: {len(plan['actions'])} - done: {len(outcome['done'])} - errors: {len(outcome['errors'])}")
for kind, relative, message in outcome["errors"]:
    print(f"  {kind} {relative} -> {message}")

# The file is put back so that the scenario is the same for the timings of section 6.
_ = shutil.move(removed_copy, source / vanished)

Actions planned: 1250 - done: 1249 - errors: 1
  copy new_00000.bin -> FileNotFoundError: No such file or directory


Delle 1250 azioni pianificate ne vengono eseguite 1249. Fallisce solo la copia del file sparito, con un `FileNotFoundError` che il riepilogo finale riporta insieme al percorso; lo stesso errore finisce nel log a livello `ERROR`. **Un file non disponibile fa fallire solo la propria azione.**

Le altre 1249 azioni non dipendono da quel file, quindi non c'è motivo di fermarsi. Se il file torna nella sorgente, al passaggio successivo risulterà assente in destinazione e verrà copiato.

Con le cartelle vale il contrario: un errore deve fermare tutto. Se la sorgente non esiste, `os.walk` non segnala niente e restituisce un elenco vuoto, quindi per il confronto ogni file della destinazione diventa obsoleto e da rimuovere. Un errore di battitura nel percorso costerebbe l'intero backup. Per questo `plan_sync`, prima di elencare qualsiasi file, chiama `check_folders`, che solleva un `ValueError` in quattro casi: sorgente mancante, sorgente che è un file, destinazione che è un file, due cartelle che si sovrappongono (la stessa cartella, o una dentro l'altra).

Le rimozioni si possono anche vedere prima di farle: con `dry_run=True` `sync_folders` restituisce la lista delle azioni senza eseguirla. Sono tre righe di codice, perché la lista esiste comunque prima dell'esecuzione. Anche rsync ha le stesse due prudenze: con `--delete` non cancella niente se ha avuto errori nel leggere la sorgente, e con `--dry-run` mostra cosa farebbe senza farlo.

In [8]:
# proviamo un percorso sbagliato, due coppie di cartelle sovrapposte e una prova a vuoto

source, destination = prepare_destination("many_files")
files_before = list_files(destination)
# One character missing from the scenario name.
mistyped = WORKSPACE / "many_file" / "source"
# What the comparison would see without check_folders: an empty source, so every file of
# the destination would be obsolete.
print(f"Deletions an unchecked comparison would plan: {len(files_before - list_files(mistyped))}")

invalid_pairs = {"mistyped source": (mistyped, destination),
                 "destination inside source": (source, source / "backup"),
                 "same folder": (source, source)}
for label, (pair_source, pair_destination) in invalid_pairs.items():
    try:
        sync_folders(pair_source, pair_destination)
    except ValueError as error:
        print(f"  {label}: refused - {error}")
    else:
        raise AssertionError(f"{label}: the sync was not refused")

preview = sync_folders(source, destination, dry_run=True)
assert list_files(destination) == files_before, "the destination was touched"
print(f"Dry run: {dict(preview['planned'])} - destination files still there: {len(list_files(destination))}")

Deletions an unchecked comparison would plan: 4750
  mistyped source: refused - source folder does not exist: /tmp/sync_lab/many_file/source
  destination inside source: refused - source and destination overlap: /tmp/sync_lab/many_files/source, /tmp/sync_lab/many_files/source/backup
  same folder: refused - source and destination overlap: /tmp/sync_lab/many_files/source, /tmp/sync_lab/many_files/source


Dry run: {'update': 500, 'copy': 500, 'delete': 250} - destination files still there: 4750


Basta un carattere mancante nel nome dello scenario: **senza il controllo, il confronto pianificherebbe la rimozione di tutti i 4750 file della destinazione**. Il controllo invece rifiuta la sorgente sbagliata, la destinazione dentro la sorgente e la cartella ripetuta prima di elencare un solo file, e il messaggio dice quale percorso non va.

La prova a vuoto elenca le stesse 1250 azioni della prova precedente, 500 copie, 500 aggiornamenti e 250 rimozioni, senza eseguirne nessuna: alla fine la destinazione ha ancora tutti i suoi 4750 file, quindi né le coppie rifiutate né la prova a vuoto l'hanno toccata.

## 6. Misura dei tempi sui tre scenari

**Obiettivo.** Misurare quanto si guadagna con il parallelismo, e in quale fase, confrontando la versione sequenziale con quella parallela su ciascuno dei tre scenari.

Misuriamo le due fasi separatamente, ognuna in tre modalità: sequenziale, thread e processi. Così vediamo se il parallelismo conviene e anche quale strumento conviene in ogni fase. Per ogni misura teniamo la migliore di cinque ripetizioni, per togliere il rumore di un'esecuzione disturbata da altro. Prima di ogni esecuzione ricreiamo la destinazione, così ogni ripetizione copia gli stessi file. Il confronto misurato è quello del motore, con le sole chiamate a `os.stat`: le impronte MD5 della sezione 4 restano fuori dai tempi.

I tempi li prendiamo con `time.perf_counter` dentro un decoratore, così lo stesso misuratore vale per entrambe le fasi. Misuriamo a parte anche l'elenco dei due alberi, la parte del confronto che resta sempre nel processo principale.

In [9]:
# misuriamo ogni fase in ogni modalità su ogni scenario

MODES = ["sequential", "threads", "processes"]
REPEATS = 5


def timed(function):
    """Decorate function so that it returns its elapsed wall-clock time instead of its result."""
    @wraps(function)
    def wrapper(*args, **kwargs):
        """Run the decorated function and return the seconds it took."""
        start = time.perf_counter()
        function(*args, **kwargs)
        return time.perf_counter() - start
    return wrapper


def best_time(function, setup):
    """Return the best of REPEATS timings of function, running setup untimed before each one."""
    timings = []
    for _ in range(REPEATS):
        arguments = setup()
        timings.append(timed(function)(*arguments))
    return min(timings)


def list_both(source, destination):
    """List the two trees: the part of every comparison that always runs in the main process."""
    return list_files(source), list_files(destination)


timing_rows = []
for name in SCENARIOS:
    source, destination = prepare_destination(name)
    actions = plan_sync(source, destination, "sequential")["actions"]
    for mode in MODES:
        # A lambda reads the loop variables when it is called, not when it is defined. Here
        # best_time calls it inside the same iteration, so the values would be right anyway;
        # the default arguments bind them explicitly, so that it no longer depends on when.
        analysis = best_time(partial(plan_sync, mode=mode),
                             lambda source=source, destination=destination: (source, destination))
        execution = best_time(partial(execute_plan, mode=mode),
                              lambda name=name, actions=actions: (*prepare_destination(name), actions))
        timing_rows.append({"scenario": name, "mode": mode, "analysis_s": analysis, "execution_s": execution})

timings = pd.DataFrame(timing_rows).set_index(["scenario", "mode"])
# Speedup over the sequential run of the same scenario: above 1 the mode is faster.
sequential = timings.xs("sequential", level="mode")
for phase in ["analysis", "execution"]:
    timings[f"{phase}_speedup"] = timings[f"{phase}_s"].rdiv(sequential[f"{phase}_s"], level="scenario")

# Whatever share of the sequential comparison the listing takes, no mode can split it.
for name in SCENARIOS:
    listing = best_time(list_both, partial(prepare_destination, name))
    share = listing / timings.loc[(name, "sequential"), "analysis_s"]
    print(f"{name} - listing the two trees: {listing:.3f} s, {share:.0%} of the sequential comparison")
timings.round(3)

few_files - listing the two trees: 0.001 s, 83% of the sequential comparison


many_files - listing the two trees: 0.403 s, 76% of the sequential comparison


nested_folders - listing the two trees: 0.259 s, 80% of the sequential comparison


analysis_s  execution_s  analysis_speedup  \
scenario       mode                                                    
few_files      sequential       0.001        0.002             1.000   
               threads          0.003        0.004             0.412   
               processes        0.015        0.030             0.087   
many_files     sequential       0.533        0.255             1.000   
               threads          0.842        0.434             0.633   
               processes        0.491        0.190             1.086   
nested_folders sequential       0.324        0.259             1.000   
               threads          0.439        0.259             0.739   
               processes        0.313        0.229             1.035   

                           execution_speedup  
scenario       mode                           
few_files      sequential              1.000  
               threads                 0.389  
               processes               0.054  
many_files     sequential              1.000  
               threads                 0.587  
               processes               1.343  
nested_folders sequential              1.000  
               threads                 0.999  
               processes               1.130

**Nel confronto il parallelismo non ripaga: i processi restano vicini alla parità con il sequenziale e i thread vanno più piano.** Le righe sopra la tabella dicono perché. L'elenco dei due alberi, che gira sempre nel processo principale, occupa la maggior parte del confronto sequenziale, e per la legge di Amdahl quella parte fissa un tetto al guadagno di qualsiasi modalità. Da dividere resta solo il lavoro per file, due chiamate a `os.stat` e poco altro: troppo poco perché due interpreti ripaghino il costo di avviarli e di rimandare indietro le azioni. I thread, invece, si contendono il GIL, e a ogni `os.stat` devono rilasciarlo e riprenderlo. È anche il motivo per cui l'MD5 sta fuori dal confronto: un lavoro in più su ogni file avrebbe fatto sembrare i processi più utili di quanto sono.

**Nell'esecuzione, su disco locale, i thread non convengono.** Se il collo di bottiglia fosse l'attesa del disco, i thread accelererebbero le copie. Invece con molti file sono più lenti del sequenziale, e con le sottocartelle il guadagno non è stabile: il rapporto resta intorno a 1 o sotto. I file degli scenari sono appena stati scritti e stanno ancora nella cache del sistema operativo, quindi la copia non aspetta quasi mai il disco e quello che pesa è il lavoro di Python su ogni file. I processi dividono questo lavoro fra due interpreti e con molti file guadagnano; con le sottocartelle, dove le azioni da eseguire sono meno della metà, anche loro restano vicini alla parità.

**Con pochi file avviare i processi costa più del lavoro da fare**, ed è l'effetto che la consegna chiedeva di mostrare: con dieci file l'esecuzione con i processi ci mette molte volte il tempo del sequenziale, con un rapporto ben sotto 0,2. Anche i thread vanno più piano, ma su tempi di pochi millisecondi, che stanno dentro il rumore della misura.

Ci sono due cautele. I tempi vanno da qualche millisecondo con pochi file a qualche decimo di secondo con gli altri, e dipendono dalla macchina e dal momento in cui si esegue: per questo nel testo riportiamo solo il verso dei confronti e lasciamo i valori alla tabella. Rieseguendo il notebook i rapporti si spostano di qualche decimo; quelli che abbiamo descritto come vicini alla parità possono anche cambiare verso, quelli netti no. Inoltre una parte del lavoro resta sequenziale in ogni modalità (l'elenco dei file, il log, la rimozione delle cartelle vuote) e, per la legge di Amdahl, limita il guadagno massimo.

In [10]:
# confrontiamo la sincronizzazione tutta sequenziale con quella scelta: processi e poi thread

totals = pd.DataFrame({
    "sequential_s": timings["analysis_s"].xs("sequential", level="mode") + timings["execution_s"].xs("sequential", level="mode"),
    "parallel_s": timings["analysis_s"].xs("processes", level="mode") + timings["execution_s"].xs("threads", level="mode"),
})
totals["speedup"] = totals["sequential_s"] / totals["parallel_s"]
totals.round(3)

,sequential_s,parallel_s,speedup
scenario,,,
few_files,0.003,0.019,0.155
many_files,0.788,0.925,0.852
nested_folders,0.583,0.572,1.019


Con la combinazione scelta (processi per il confronto, thread per l'esecuzione) la sincronizzazione completa su questo disco non guadagna: sugli scenari grandi resta vicina alla parità o sotto, e con pochi file è molto più lenta.

**Su questo disco, per l'esecuzione, i thread non sono la scelta migliore in nessuno scenario**, come si vede nella tabella precedente: sugli scenari grandi vanno meglio i processi o, quasi alla pari, il sequenziale.

## 7. Conclusioni

### 7.1 Cosa abbiamo osservato

Su un runtime con due CPU una sincronizzazione completa dura al massimo qualche centesimo di secondo con pochi file e qualche decimo di secondo con molti file o con le sottocartelle (sezione 6). Con questi tempi:

- **il motore è corretto su tutti e tre gli scenari**: le impronte in destinazione coincidono con quelle della sorgente e un secondo passaggio non trova azioni (sezione 4);
- **un file non disponibile non ferma la sincronizzazione**: l'errore viene raccolto, riportato alla fine e scritto nel log (sezione 5);
- **una coppia di cartelle sbagliata ferma la sincronizzazione prima che tocchi un file**: senza il controllo, una sorgente inesistente verrebbe letta come vuota e la destinazione verrebbe svuotata (sezione 5);
- **nel confronto il parallelismo non ripaga**: l'elenco dei due alberi resta sequenziale e il lavoro per file è leggero, quindi i processi restano vicini alla parità e i thread, frenati dal GIL, sono più lenti del sequenziale (sezione 6);
- **per l'esecuzione i thread convengono solo se la scrittura deve aspettare**: su disco locale non accelerano (sezione 6), con una latenza su ogni scrittura diventano di gran lunga la modalità più veloce (sezione 8);
- **con pochi file i processi non convengono**: avviarli costa più del lavoro da fare, e la sincronizzazione completa è più lenta di quella sequenziale (sezione 6).

### 7.2 Raccomandazioni per il team infrastruttura

- Per cartelle piccole come quella dello scenario con pochi file conviene fare tutto in sequenziale. La soglia sta fra i dieci file e le migliaia degli altri scenari e non l'abbiamo misurata; `sync_folders` riceve le due modalità come parametri, quindi cambiarle non richiede di toccare il codice.
- Su una destinazione nuova, lanciare la prima sincronizzazione con `dry_run=True` e controllare le rimozioni previste prima di eseguirle.
- Verso una destinazione di rete o cloud, usare i thread per l'esecuzione e sceglierne il numero in base alla latenza reale, non al numero di CPU.
- Verso un disco locale con molti file piccoli, usare i processi anche per l'esecuzione.
- Su disco locale il confronto può restare sequenziale: i processi non lo accelerano in modo apprezzabile.

## 8. Approfondimento oltre la consegna: una destinazione con latenza

**Obiettivo.** Capire in quali condizioni i thread accelerano davvero l'esecuzione, visto che su disco locale non lo fanno (sezione 6).

La destinazione di un backup spesso è una cartella di rete o uno storage cloud, dove ogni scrittura aspetta la risposta di un server. Simuliamo l'attesa con una pausa fissa prima di ogni copia, `WRITE_LATENCY_S`, e rifacciamo la misura dell'esecuzione sullo scenario con molti file.

Scriviamo la previsione prima di misurare. Se i thread guadagnassero grazie alle CPU, si fermerebbero vicino al numero di CPU, come i processi. Se invece guadagnano perché sovrappongono le attese, devono avvicinarsi al numero di thread.

In [11]:
# simuliamo una destinazione remota aggiungendo un'attesa fissa a ogni scrittura

# Rough model of the round trip of a backup on a network share or a cloud bucket: a fixed
# wait per written file, during which the CPU has nothing to do. The value is an assumption.
WRITE_LATENCY_S = 0.002


def remote_copy(source_path, target_path):
    """Copy like shutil.copy2, after waiting WRITE_LATENCY_S as a remote destination would."""
    time.sleep(WRITE_LATENCY_S)
    shutil.copy2(source_path, target_path)


name = "many_files"
source, destination = prepare_destination(name)
actions = plan_sync(source, destination, "sequential")["actions"]
remote_rows = []
for mode in MODES:
    execution = best_time(partial(execute_plan, mode=mode, copy_function=remote_copy),
                          lambda: (*prepare_destination(name), actions))
    remote_rows.append({"mode": mode, "execution_s": execution})

remote = pd.DataFrame(remote_rows).set_index("mode")
remote["speedup"] = remote.loc["sequential", "execution_s"] / remote["execution_s"]
remote.round(3)

,execution_s,speedup
mode,,
sequential,2.709,1.000
threads,0.452,5.997
processes,1.409,1.923


La previsione è confermata: **con la latenza i thread rendono l'esecuzione circa 6 volte più veloce del sequenziale**, mentre i processi si fermano a circa 2, vicino al numero di CPU. Durante la pausa un thread non usa la CPU e rilascia il GIL, quindi otto thread tengono aperte otto attese insieme, mentre due processi ne tengono aperte solo due. Il guadagno resta sotto il massimo teorico di otto per la parte del lavoro che non aspetta: la copia vera e propria, il log e la creazione delle cartelle.

La latenza qui è simulata, quindi con una destinazione reale il guadagno esatto sarà diverso, ma il verso del confronto resta lo stesso. **Questo risultato giustifica la scelta della sezione 3: in un caso di backup verso una destinazione remota l'esecuzione va affidata ai thread.**

---

# NOTE DI PROGETTO: scelte e limiti di sviluppo

Qui sono raccolte le decisioni prese durante lo sviluppo e i limiti noti del prototipo, per tenere separato quello che abbiamo scelto da quello che sarebbe un difetto.

## Scelte di progetto

**1. Confronto ed esecuzione separati.**
Uno script improvvisato copia mentre scorre le cartelle. Noi prima costruiamo la lista delle azioni e poi la eseguiamo: la lista si può stampare e controllare (sezione 2), si divide in sottogruppi indipendenti e ogni fase gira nella modalità più adatta. Il prezzo è che fra le due fasi un file può cambiare o sparire; lo gestiamo come errore di quel singolo file (sezione 5).

**2. Il timestamp decide, l'impronta verifica.**
Confrontare il contenuto con un hash sarebbe più sicuro, ma bisognerebbe leggere per intero anche la destinazione, e su un backup remoto costa molto. Per decidere usiamo quindi solo il timestamp, come chiede la consegna, e l'MD5 della sorgente ci serve solo per verificare il risultato (sezione 4). Lo calcoliamo nel codice di prova, fuori dal motore, così i tempi della sezione 6 misurano solo il lavoro di una sincronizzazione vera, che non rilegge il contenuto dei file. La dimensione non entra nella regola, perché i file modificati della sezione 1 hanno la stessa dimensione dell'originale.

**3. Cartelle controllate prima del confronto.**
Trattare ogni problema come errore di un singolo file, raccolto e riportato alla fine, sarebbe coerente con la sezione 5, ma una sorgente mancante non dà nessun errore: `os.walk` la legge come vuota. Il controllo va quindi fatto prima, e a differenza degli altri errori ferma tutto. Il prezzo è che una sorgente irraggiungibile per poco, come una cartella di rete che non risponde, fa saltare l'intera sincronizzazione invece di rimandarla di un file.

**4. Rimozioni prima delle copie.**
Un solo turno con tutte le azioni sarebbe più semplice e avvierebbe i worker una volta sola, ma lascerebbe il conflitto fra file e cartella con lo stesso nome all'ordine casuale dei worker (sezione 4). Il prezzo del secondo turno si vede sulle cartelle piccole, dove l'avvio dei processi pesa già più del lavoro.

**5. Processi per il confronto, thread per l'esecuzione.**
Il confronto è lavoro dell'interprete su ogni file, e solo i processi lo eseguono davvero in parallelo. Con le sole chiamate a `os.stat` è però un lavoro leggero, e l'elenco dei due alberi resta comunque sequenziale: sugli scenari grandi i processi restano vicini alla parità e i thread perdono. Li lasciamo come default perché sulle cartelle grandi non perdono, e la scelta si cambia con un parametro (sezione 7.2). Per l'esecuzione la sezione 6 mostra che su disco locale i thread non accelerano e non sono mai la modalità più veloce. Li teniamo perché sono la scelta giusta verso una destinazione che deve aspettare (sezione 8). Il prezzo è un po' di guadagno perso quando la destinazione è un disco locale.

**6. Sottogruppi distribuiti a rotazione.**
Dividere per cartella di primo livello sarebbe più intuitivo, ma con molti file in un'unica cartella tutto il lavoro finirebbe a un solo worker. Distribuendo i file a rotazione, ogni worker riceve più o meno la stessa quota di file piccoli e grandi.

**7. Log scritto solo dal processo principale.**
Per far scrivere il log direttamente ai worker servirebbe una coda condivisa fra processi. I worker invece restituiscono l'esito delle azioni e il processo principale lo registra, indicando in ogni riga quale worker ha eseguito l'azione. Il prezzo è che il log di una fase si scrive alla fine della fase e non in tempo reale.

**8. Processi avviati con `fork`, dichiarato esplicitamente.**
Con `fork` i processi figli ereditano le funzioni definite nel notebook. Con `spawn` e `forkserver` partono invece da un interprete nuovo, che quelle funzioni non le vede. Lo dichiariamo esplicitamente perché da Python 3.14 il default su Linux cambia, e così il comportamento non dipende dalla versione.

## Analisi aggiunte oltre la consegna

**1. Destinazione con latenza (sezione 8).**
La sezione 6 mostra che su disco locale i thread non accelerano l'esecuzione, e questo sembra contraddire la scelta della sezione 3. L'approfondimento mostra in quali condizioni quella scelta è corretta e ci permette di trasformarla in una raccomandazione motivata (sezione 7.2).

## Limiti riscontrati

**1. Tempi misurati su una sola macchina.** Due CPU e un disco servito dalla memoria del sistema operativo: in un altro ambiente i rapporti cambiano, e rieseguendo il notebook cambiano anche i valori stampati. Per questo le conclusioni si basano sul verso dei confronti e non sulle cifre esatte.

**2. Viene propagata solo la sorgente più recente.** Un file modificato a mano in destinazione dopo l'ultima sincronizzazione ha un timestamp più recente della sorgente, quindi non viene riallineato. Il confronto presuppone anche che gli orologi delle due macchine siano coerenti.

**3. Il motore non verifica le proprie copie.** L'MD5 serve solo alla prova della sezione 4. In produzione una copia rovinata durante la scrittura passerebbe inosservata, e siccome `copy2` le dà lo stesso timestamp della sorgente, i passaggi successivi la considererebbero aggiornata.

**4. Solo file regolari.** Le cartelle vuote presenti in sorgente non vengono create in destinazione, e collegamenti simbolici e permessi non hanno una gestione specifica.

**5. Non funziona su Windows.** Su Windows i processi partono solo con `spawn`, che non vede le funzioni definite in una cella: per usarlo lì bisognerebbe spostare il motore in un modulo importabile. Lo script che lo lancia dovrebbe poi avviare la sincronizzazione sotto la guardia `if __name__ == "__main__":`, perché con `spawn` ogni processo figlio importa lo script principale e senza guardia lo rieseguirebbe da capo.

**6. Tempi misurati con il GIL attivo.** La sezione 0 mostra che il runtime usa la build standard di CPython, con il GIL. Da Python 3.13 esiste anche una build senza GIL (free-threaded), in cui i thread eseguono in parallelo anche il lavoro dell'interprete: lì il confronto fra thread e processi della sezione 6 andrebbe rifatto da capo.